In [1]:
import gc
import os
import sys
from pathlib import Path

import torch
from datasets import load_dataset

# Make sure we can import train1.py from this notebook directory.
NOTEBOOK_DIR = Path.cwd()
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.append(str(NOTEBOOK_DIR))

from train1 import GPTConfig, miniGPT

/Users/yanfu/Desktop/School/2026/TeenyGPT/stupidcookgpt/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Build a smaller GPT profile to avoid MPS OOM on Apple Silicon.
config = GPTConfig(
    block_size=484,
    n_layers=12,
    n_embd=768,
    n_heads=12,
    dropout=0.1,
)

trainer = miniGPT.from_tokenizer_name("gpt2", config=config)

device = "cpu" if torch.backends.mps.is_available() else ("cuda" if torch.cuda.is_available() else "cpu")
trainer = trainer.to(device)
print(f"Using device: {device}")

Using device: cpu


In [3]:
# Load huggingface/all-recipes and split exactly as requested.
raw = load_dataset("corbt/all-recipes")

# Some configs expose only a train split; this handles both cases safely.
if "train" in raw:
    base_split = raw["train"]
else:
    first_key = list(raw.keys())[0]
    base_split = raw[first_key]

split = base_split.train_test_split(test_size=0.05, seed=42)
train_full = split["train"]
test_set = split["test"]

print(f"train_full: {len(train_full):,} samples")
print(f"test_set: {len(test_set):,} samples")

train_full: 2,039,885 samples
test_set: 107,363 samples


In [4]:
# Make a smaller training subset so experiments run faster.
subset_size = min(40000, len(train_full))
subset_seed = 42
train_subset = train_full.shuffle(seed=subset_seed).select(range(subset_size))

# Pick a likely text field from the dataset.
candidate_columns = [
    "text",
    "recipe",
    "instructions",
    "directions",
    "title",
    "ingredients",
    "source",
    "input",
]

available_columns = train_subset.column_names
text_column = next((c for c in candidate_columns if c in available_columns), None)
if text_column is None and len(available_columns) == 1:
    text_column = available_columns[0]
if text_column is None:
    raise ValueError(f"No expected text column found. Available: {available_columns}")

def _flatten_to_text(value):
    if value is None:
        return ""
    if isinstance(value, str):
        return value
    if isinstance(value, list):
        return " ".join(_flatten_to_text(v) for v in value)
    if isinstance(value, dict):
        # Prefer common content keys when present.
        for key in ["text", "input", "recipe", "instructions", "directions", "title", "ingredients"]:
            if key in value:
                return _flatten_to_text(value[key])
        return " ".join(f"{k}: {_flatten_to_text(v)}" for k, v in value.items())
    return str(value)

def to_text(row):
    return _flatten_to_text(row.get(text_column)).strip()

train_texts = [to_text(r) for r in train_subset]
train_texts = [t for t in train_texts if t]

print(f"Selected text column: {text_column}")
print(f"train_subset used: {len(train_texts):,} samples")

Selected text column: input
train_subset used: 40,000 samples


In [5]:
# Clear Python and MPS cached objects before training to avoid OOM spikes.
gc.collect()
if device == "mps":
    torch.mps.empty_cache()

# Train on the smaller subset.
trainer.train(
    train_texts,
    epochs=1,
    batch_size=8,
    learning_rate=3e-4,
    warmup_ratio=0.03,
    min_lr_ratio=0.1,
)

# Save pure weights plus the tokenizer and config metadata.
output_dir = NOTEBOOK_DIR / "trained_model_small_5k"
trainer.save(str(output_dir))

print(f"Saved weights to: {output_dir / 'weights.pt'}")
print(f"Saved config to: {output_dir / 'config.json'}")

Epoch 1/1, Batch 1/5000, Loss: 10.9763, LR: 0.00e+00
Epoch 1/1, Batch 2/5000, Loss: 10.9695, LR: 2.00e-06
Epoch 1/1, Batch 3/5000, Loss: 10.9664, LR: 4.00e-06
Epoch 1/1, Batch 4/5000, Loss: 10.9177, LR: 6.00e-06
Epoch 1/1, Batch 5/5000, Loss: 10.8567, LR: 8.00e-06
Epoch 1/1, Batch 6/5000, Loss: 10.7894, LR: 1.00e-05
Epoch 1/1, Batch 7/5000, Loss: 10.7014, LR: 1.20e-05
Epoch 1/1, Batch 8/5000, Loss: 10.6174, LR: 1.40e-05
Epoch 1/1, Batch 9/5000, Loss: 10.5249, LR: 1.60e-05
Epoch 1/1, Batch 10/5000, Loss: 10.4350, LR: 1.80e-05
Epoch 1/1, Batch 11/5000, Loss: 10.3442, LR: 2.00e-05
Epoch 1/1, Batch 12/5000, Loss: 10.2667, LR: 2.20e-05
Epoch 1/1, Batch 13/5000, Loss: 10.1760, LR: 2.40e-05
Epoch 1/1, Batch 14/5000, Loss: 10.1110, LR: 2.60e-05
Epoch 1/1, Batch 15/5000, Loss: 10.0400, LR: 2.80e-05
Epoch 1/1, Batch 16/5000, Loss: 9.9710, LR: 3.00e-05
Epoch 1/1, Batch 17/5000, Loss: 9.9092, LR: 3.20e-05
Epoch 1/1, Batch 18/5000, Loss: 9.8556, LR: 3.40e-05
Epoch 1/1, Batch 19/5000, Loss: 9.8001, 

In [13]:
output_dir = NOTEBOOK_DIR / "trained_model_small_5k"

# Load the trained model and generate some recipes.
loaded_trainer = miniGPT.load(str(output_dir), map_location=device)

# Generate a few recipe samples with different prompts.
prompts = [
    "Recipe for chocolate ",
    "Ingredients: flour butter sugar",
    "Instructions: preheat oven",
    "Recipe for chocolate",
    "Recipe for chocolate",
    "Recipe for",
    "Gradually stir in",
]

print("=" * 60)
print("GENERATED RECIPES")
print("=" * 60)

for prompt in prompts:
    print(f"\nPrompt: {prompt}")
    print("-" * 60)
    generated = loaded_trainer.generate(prompt, max_length=250, temperature=0.8, top_p=0.95)
    print(generated)
    print()

GENERATED RECIPES

Prompt: Recipe for chocolate 
------------------------------------------------------------
Recipe for chocolate 

Ingredients:
- 1 1/2 cup white chocolate morsels
- 1/2 cup butter, melted
- 1 cup confectioners' sugar
- 3/4 cup brown sugar
- 1 cup sugar
- 2 tablespoons vanilla extract
- 2 teaspoons vanilla extract
- 4 large eggs
- 2 tablespoons vanilla
- 1 cup milk
- 2 1/2 cups semi-sweet chocolate chips
- 1 teaspoon vanilla

Directions:
- Grease a 9x9 inch pan.
- Mix ingredients together except chocolate and press over chocolate mixture.
- Bake at 350° for 15 minutes. Remove from pan.
- Remove from oven and cool completely. Cool completely.
- Melt chocolate in saucepan over low heat and stir constantly. Remove from heat. Remove from heat. Add peanut butter and vanilla. Pour into pan.
- Bake for about 15 minutes or until set.
- To serve, add chocolate, chocolate chips, chocolate chips, chocolate chips and marshmallows.
- Cool before serving.


Prompt: Ingredients: flo

In [ ]:
# Calculate total parameters in the loaded model
total_params = sum(p.numel() for p in loaded_trainer.model.parameters())
trainable_params = sum(p.numel() for p in loaded_trainer.model.parameters() if p.requires_grad)
print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
# print out model name
print(f"Model name: miniGPT")

Total parameters: 124,025,088
Trainable parameters: 124,025,088
Model name: GPTModel
